In [ ]:
# ============================================================
# Hücre 1 — Kütüphaneler
# ============================================================
from google.colab import files
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

In [ ]:
# ============================================================
# Hücre 2 — Dosya yükleme
# ============================================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print("Uploaded:", file_name)

Saving train_data.txt to train_data.txt
Uploaded: train_data.txt


In [ ]:
# ============================================================
# Hücre 3 — Veri okuma
# ============================================================
feature_names = [
    "Jitter_local", "Jitter_local_abs", "Jitter_rap", "Jitter_ppq5", "Jitter_ddp",
    "Shimmer_local", "Shimmer_local_dB", "Shimmer_apq3", "Shimmer_apq5", "Shimmer_apq11", "Shimmer_dda",
    "AC", "NTH", "HTN",
    "Median_pitch", "Mean_pitch", "Pitch_std", "Min_pitch", "Max_pitch",
    "Num_pulses", "Num_periods", "Mean_period", "Period_std",
    "Frac_unvoiced_frames", "Num_voice_breaks", "Degree_voice_breaks",
]
cols = ["SubjectId"] + feature_names + ["UPDRS", "Class"]

df = pd.read_csv(file_name, header=None, names=cols)
df.head()

,SubjectId,Jitter_local,Jitter_local_abs,Jitter_rap,Jitter_ppq5,Jitter_ddp,Shimmer_local,Shimmer_local_dB,Shimmer_apq3,Shimmer_apq5,...,Max_pitch,Num_pulses,Num_periods,Mean_period,Period_std,Frac_unvoiced_frames,Num_voice_breaks,Degree_voice_breaks,UPDRS,Class
0,1,1.488,0.000090,0.900,0.794,2.699,8.334,0.779,4.517,4.609,...,187.576,160,159,0.006065,0.000416,0.000,0,0.000,23,1
1,1,0.728,0.000038,0.353,0.376,1.059,5.864,0.642,2.058,3.180,...,234.505,170,169,0.005181,0.000403,2.247,0,0.000,23,1
2,1,1.220,0.000074,0.732,0.670,2.196,8.719,0.875,4.347,5.166,...,211.442,1431,1427,0.006071,0.000474,10.656,1,0.178,23,1
3,1,2.502,0.000123,1.156,1.634,3.469,13.513,1.273,5.263,8.771,...,220.230,94,92,0.004910,0.000320,0.000,0,0.000,23,1
4,1,3.509,0.000167,1.715,1.539,5.145,9.112,1.040,3.102,4.927,...,225.162,117,114,0.004757,0.000380,18.182,1,13.318,23,1


In [ ]:
# ============================================================
# Hücre 4 — Özet metrik fonksiyonları
# ============================================================

def trimmed_mean_25(x):
    return stats.trim_mean(x, 0.25)

def median_abs_dev(x):
    return np.median(np.abs(x - np.median(x)))

def p90_p10(x):
    return np.percentile(x, 90) - np.percentile(x, 10)

central_funcs = {
    "mean": lambda x: np.mean(x),
    "median": lambda x: np.median(x),
    "trimmed_mean_25": trimmed_mean_25,
}

dispersion_funcs = {
    "std": lambda x: np.std(x, ddof=1),
    "IQR": lambda x: stats.iqr(x),
    "MedAD": median_abs_dev,
    "p90_p10": p90_p10,
}

In [ ]:
# ============================================================
# Hücre 5 — Seçilen 5 kombinasyon
# ============================================================

combinations = [
    ("mean", "std"),
    ("trimmed_mean_25", "IQR"),
    ("median", "IQR"),
    ("median", "MedAD"),
    ("median", "p90_p10"),
]

In [ ]:
# ============================================================
# Hücre 6 — Subject-level özet üretici
# ============================================================

def create_subject_dataset(df, central_name, dispersion_name):
    rows = []

    for subject, group in df.groupby("SubjectId"):
        feats = group[feature_names].values

        c = np.apply_along_axis(central_funcs[central_name], 0, feats)
        d = np.apply_along_axis(dispersion_funcs[dispersion_name], 0, feats)

        combined = np.concatenate([c, d])
        label = group["Class"].iloc[0]

        rows.append(np.append(combined, label))

    columns = [f"{f}_{central_name}" for f in feature_names] + \
              [f"{f}_{dispersion_name}" for f in feature_names] + \
              ["Class"]

    return pd.DataFrame(rows, columns=columns)

In [ ]:
# ============================================================
# Hücre 7 — LOO değerlendirme fonksiyonu
# ============================================================

def evaluate_model(X, y, model):
    loo = LeaveOneOut()
    y_true, y_pred = [], []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        y_true.append(int(y_test.values[0]))
        y_pred.append(int(pred[0]))

    acc = accuracy_score(y_true, y_pred)
    sens = recall_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp)

    return acc, sens, spec

In [ ]:
# ============================================================
# Hücre 8 — Tüm kombinasyonları çalıştır
# ============================================================

results = []

for central, dispersion in combinations:

    subject_df = create_subject_dataset(df, central, dispersion)

    X = subject_df.drop(columns=["Class"])
    y = subject_df["Class"]

    # Linear SVM
    acc_l, sens_l, spec_l = evaluate_model(
        X, y, SVC(kernel="linear", C=1)
    )

    # RBF SVM
    acc_r, sens_r, spec_r = evaluate_model(
        X, y, SVC(kernel="rbf", C=10, gamma=0.01)
    )

    results.append([f"{central}+{dispersion}",
                    acc_l, sens_l, spec_l,
                    acc_r, sens_r, spec_r])

results_df = pd.DataFrame(results, columns=[
    "Combination",
    "Linear_Acc", "Linear_Sens", "Linear_Spec",
    "RBF_Acc", "RBF_Sens", "RBF_Spec"
])

results_df

,Combination,Linear_Acc,Linear_Sens,Linear_Spec,RBF_Acc,RBF_Sens,RBF_Spec
0,mean+std,0.775,0.80,0.75,0.600,0.60,0.6
1,trimmed_mean_25+IQR,0.650,0.65,0.65,0.700,0.70,0.7
2,median+IQR,0.675,0.60,0.75,0.700,0.70,0.7
3,median+MedAD,0.700,0.70,0.70,0.775,0.75,0.8
4,median+p90_p10,0.550,0.60,0.50,0.550,0.60,0.5


In [ ]:
# median + MedAD benim katkım rbf svm de iyi sonuç verdi

In [ ]:
def mean_abs_dev(x):
    return np.mean(np.abs(x - np.mean(x)))

In [ ]:
test_combinations = [
    ("median", "MeanAD"),
    ("median", "MedAD"),
]

In [ ]:
dispersion_funcs["MeanAD"] = mean_abs_dev

In [ ]:
results = []

for central, dispersion in test_combinations:

    subject_df = create_subject_dataset(df, central, dispersion)

    X = subject_df.drop(columns=["Class"])
    y = subject_df["Class"]

    acc, sens, spec = evaluate_model(
        X, y, SVC(kernel="rbf", C=10, gamma=0.01)
    )

    results.append([f"{central}+{dispersion}", acc, sens, spec])

pd.DataFrame(results, columns=["Combination", "RBF_Acc", "RBF_Sens", "RBF_Spec"])

,Combination,RBF_Acc,RBF_Sens,RBF_Spec
0,median+MeanAD,0.600,0.60,0.6
1,median+MedAD,0.775,0.75,0.8


In [ ]:
# Parkinson ses sınıflandırmasında, RBF tabanlı modeller için robust dispersion metriği (MedAD), klasik mean absolute deviation’a kıyasla belirgin performans artışı sağlamıştır.